# 構造化出力とエージェンティックワークフロー


In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## 構造化出力（Structured Outputs）


### Chat Completions APIの構造化出力


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge(inquiry: str) -> SpamJudgeResult:
    client = OpenAI()
    response = client.chat.completions.parse(
        model="gpt-5-nano",
        messages=[
            {
                "role": "developer",
                "content": "以下のお問い合わせがスパムか判定してください。",
            },
            {
                "role": "user",
                "content": f"お問い合わせ: {inquiry}",
            },
        ],
        reasoning_effort="minimal",
        response_format=SpamJudgeResult,
    )

    return response.choices[0].message.parsed


result = spam_judge("AIエージェントの開発を依頼したいです。")
print(type(result))
print(result.model_dump_json(indent=2))


### LangChainのwith_structured_outputを使った構造化出力


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge(inquiry: str) -> SpamJudgeResult:
    model = init_chat_model(
        model="gpt-5-nano",
        model_provider="openai",
        reasoning_effort="minimal",
    )
    model_with_structure = model.with_structured_output(SpamJudgeResult)

    prompt = [
        SystemMessage(content="以下のお問い合わせがスパムか判定してください。"),
        HumanMessage(content=f"お問い合わせ: {inquiry}"),
    ]
    return model_with_structure.invoke(prompt)


result = spam_judge("AIエージェントの開発を依頼したいです。")
print(type(result))
print(result.model_dump_json(indent=2))

## 返信文生成ワークフロー


In [ ]:
from typing_extensions import TypedDict


class State(TypedDict):
    # 入力
    inquiry: str
    # スパム判定の出力
    is_spam: bool
    reason_in_japanese: str
    # 返信文生成の出力
    response: str | None

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    reasoning_effort="medium",
)

In [ ]:
from typing import Any
from pydantic import BaseModel, Field


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge_node(state: State) -> dict[str, Any]:
    model_with_structure = model.with_structured_output(SpamJudgeResult)
    prompt = [
        SystemMessage(content="以下のお問い合わせがスパムか判定してください。"),
        HumanMessage(content=f"お問い合わせ: {state['inquiry']}"),
    ]
    return model_with_structure.invoke(prompt)

In [ ]:
def generate_answer_node(state: State) -> dict[str, Any]:
    prompt = [
        SystemMessage(content="以下のお問い合わせに対する、返信文を出力してください。"),
        HumanMessage(content=f"お問い合わせ: {state['inquiry']}"),
    ]
    ai_message = model.invoke(prompt)
    return {"response": ai_message.content}


In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

graph_builder.add_node("spam_judge_node", spam_judge_node)
graph_builder.add_node("generate_answer_node", generate_answer_node)

graph_builder.add_edge(START, "spam_judge_node")

# checkノードから次のノードへの遷移に条件付きエッジを定義
# state.is_spamの値がTrueならENDノードへ、Falseならgenerate_answer_nodeへ
graph_builder.add_conditional_edges(
    "spam_judge_node",
    lambda state: state["is_spam"],
    {True: END, False: "generate_answer_node"},
)

graph_builder.add_edge("generate_answer_node", END)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
import json

initial_state = {"inquiry": "AIエージェントの開発を依頼したいです。"}
final_state = graph.invoke(initial_state)
print(json.dumps(final_state, ensure_ascii=False, indent=2))

# Tool useとAIエージェント


## Function calling


### Function callingをそのまま使う例


In [ ]:
import json


def get_current_weather(location: str, unit: str = "celsius") -> str:
    return json.dumps({"location": location, "temperature": 20, "unit": unit})

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city, e.g. Tokyo",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "default": "celsius",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "東京の現在の天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    reasoning_effort="minimal",
)
print(response.to_json(indent=2))

In [ ]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())
print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)
    print(function_response.encode("utf-8").decode("unicode_escape"))

    # 関数の実行結果を会話履歴としてmessagesに追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
second_response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    reasoning_effort="minimal",
)
print(second_response.to_json(indent=2))

### LangChainのcreate_agentと@toolでの実装


In [ ]:
from typing import Any
from langchain.tools import tool


@tool
def get_current_weather(location: str, unit: str = "celsius") -> dict[str, Any]:
    """Get the current weather in a given location"""
    return {"location": location, "temperature": 20, "unit": unit}

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    reasoning_effort="minimal",
)


agent = create_agent(
    model=model,
    tools=[get_current_weather],
)

In [ ]:
from langchain.messages import HumanMessage

initial_state = {"messages": [HumanMessage(content="東京の現在の天気はどうですか？")]}

for event in agent.stream(initial_state, stream_mode="updates"):
    for value in event.values():
        latest_message = value["messages"][-1]
        latest_message.pretty_print()
